In [1]:
import pyarrow.dataset as ds
import pandas as pd
from pathlib import Path
import numpy as np
import statsmodels.api as sm

In [6]:
ROOT = Path("/home/hanwenying")
DATA = ROOT / "rothman-data/gltd"
OUT  = ROOT / "rothman-anna/gltd/out"

In [3]:
gtex = ds.dataset(DATA / "tpm-gtex-v11.parquet", format="parquet")

In [20]:
gtexdf = gtex.head(2000).to_pandas()

In [19]:
genelengths = pd.read_csv(DATA / "gencode-v47-lengths.tsv", sep='\t')
phenotypes = pd.read_csv(DATA / "phenotypes-gtex-v11.txt", sep='\t')
attributes = pd.read_csv(DATA / "attributes-gtex-v11.txt", sep='\t')

/tmp/ipykernel_973023/2919796179.py:3: DtypeWarning: Columns (0: SMGTC) have mixed types. Specify dtype option on import or set low_memory=False.
  attributes = pd.read_csv(DATA / "attributes-gtex-v11.txt", sep='\t')


In [21]:
genelengths = pd.DataFrame(genelengths.set_index('gene')['merged'])
genelengths['logmerged'] = np.log10(genelengths['merged'])

In [70]:
metadata = pd.read_csv(OUT / "metadata.tsv", sep='\t')

In [72]:
metadata = metadata.drop(columns=['SMRIN_x', 'SMRIN_y'])

,SUBJID,SEX,AGE,SAMPID,SMTSD
0,GTEX-1117F,1,4,GTEX-1117F-0005-SM-HL9SH,Whole Blood
1,GTEX-1117F,1,4,GTEX-1117F-0011-R10b-SM-GI4VE,Brain - Frontal Cortex (BA9)
2,GTEX-1117F,1,4,GTEX-1117F-0011-R11b-SM-GIN8R,Brain - Cerebellar Hemisphere
3,GTEX-1117F,1,4,GTEX-1117F-0011-R2b-SM-GI4VL,Brain - Substantia nigra
4,GTEX-1117F,1,4,GTEX-1117F-0011-R3a-SM-GJ3PJ,Brain - Anterior cingulate cortex (BA24)
...,...,...,...,...,...
19783,GTEX-ZZPU,1,3,GTEX-ZZPU-2326-SM-GOQYU,Nerve - Tibial
19784,GTEX-ZZPU,1,3,GTEX-ZZPU-2426-SM-5E44I,Artery - Tibial
19785,GTEX-ZZPU,1,3,GTEX-ZZPU-2526-SM-GOQZ3,Skin - Sun Exposed (Lower leg)
19786,GTEX-ZZPU,1,3,GTEX-ZZPU-2626-SM-5E45Y,Muscle - Skeletal


In [74]:
df_merged = pd.merge(metadata, attributes[['SAMPID', 'SMRIN']], on='SAMPID', how='left')

In [75]:
df_merged

,SUBJID,SEX,AGE,SAMPID,SMTSD,SMRIN
0,GTEX-1117F,1,4,GTEX-1117F-0005-SM-HL9SH,Whole Blood,8.3
1,GTEX-1117F,1,4,GTEX-1117F-0011-R10b-SM-GI4VE,Brain - Frontal Cortex (BA9),7.2
2,GTEX-1117F,1,4,GTEX-1117F-0011-R11b-SM-GIN8R,Brain - Cerebellar Hemisphere,6.0
3,GTEX-1117F,1,4,GTEX-1117F-0011-R2b-SM-GI4VL,Brain - Substantia nigra,5.7
4,GTEX-1117F,1,4,GTEX-1117F-0011-R3a-SM-GJ3PJ,Brain - Anterior cingulate cortex (BA24),7.1
...,...,...,...,...,...,...
19783,GTEX-ZZPU,1,3,GTEX-ZZPU-2326-SM-GOQYU,Nerve - Tibial,5.5
19784,GTEX-ZZPU,1,3,GTEX-ZZPU-2426-SM-5E44I,Artery - Tibial,7.2
19785,GTEX-ZZPU,1,3,GTEX-ZZPU-2526-SM-GOQZ3,Skin - Sun Exposed (Lower leg),5.6
19786,GTEX-ZZPU,1,3,GTEX-ZZPU-2626-SM-5E45Y,Muscle - Skeletal,8.5


In [76]:
df_merged.to_csv(OUT / "metadata.tsv", sep='\t', index=False)

In [25]:
metadata = df_merged

In [26]:
gtexdf = gtexdf[gtexdf.index.isin(gtexdf.index.intersection(genelengths.index))].drop(columns=['Description'])

In [27]:
gtexdf

,GTEX-1117F-0005-SM-HL9SH,GTEX-1117F-0011-R10b-SM-GI4VE,GTEX-1117F-0011-R11b-SM-GIN8R,GTEX-1117F-0011-R2b-SM-GI4VL,GTEX-1117F-0011-R3a-SM-GJ3PJ,GTEX-1117F-0011-R4b-SM-GI4VM,GTEX-1117F-0011-R5a-SM-GI4VW,GTEX-1117F-0011-R6a-SM-GI4VX,GTEX-1117F-0011-R7a-SM-H65ZK,GTEX-1117F-0226-SM-5GZZ7,...,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2326-SM-GOQYU,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2526-SM-GOQZ3,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
Name,,,,,,,,,,,,,,,,,,,,,
ENSG00000290825.2,0.000000,0.000000,0.015825,0.000000,0.000000,0.034443,0.000000,0.013671,0.000000,0.000000,...,0.028982,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.016663,0.000000,0.000000
ENSG00000223972.6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000310526.1,1.163903,1.260954,3.023480,0.335984,0.631406,0.844382,1.409834,1.508202,1.038700,5.928731,...,4.863250,2.082317,1.840574,3.879934,1.826294,3.870932,1.475226,6.310629,0.703600,2.185337
ENSG00000243485.6,0.000000,0.000000,0.027433,0.000000,0.027257,0.000000,0.000000,0.023698,0.021193,0.000000,...,0.000000,0.000000,0.000000,0.059820,0.000000,0.000000,0.000000,0.000000,0.000000,0.042076
ENSG00000237613.3,0.000000,0.023562,0.000000,0.012465,0.034029,0.037269,0.000000,0.000000,0.013229,0.000000,...,0.000000,0.000000,0.021915,0.000000,0.000000,0.041812,0.000000,0.018030,0.000000,0.026265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000228929.1,0.253643,1.173384,0.262390,0.429765,0.260713,0.333124,0.256983,0.850019,0.557449,4.183819,...,1.201308,2.298405,3.106238,6.579891,4.713085,2.722916,4.267226,3.867861,1.724981,2.716570
ENSG00000206627.1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.219601,0.000000,...,0.000000,0.000000,0.000000,0.619845,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000203995.11,0.049874,0.070991,0.735207,0.065726,0.051264,0.084217,0.070742,0.144853,0.159434,0.099287,...,3.590425,0.060837,0.057776,0.281261,1.413263,0.086610,0.081860,0.794487,0.000000,0.059351


In [28]:
attributes['SUBJID'] = attributes['SAMPID'].str.extract(r'(^GTEX-[^-]+)')
attributes = attributes.dropna(subset=['SUBJID'])

/tmp/ipykernel_973023/2096518487.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  attributes['SUBJID'] = attributes['SAMPID'].str.extract(r'(^GTEX-[^-]+)')


In [29]:
metadata = pd.merge(phenotypes, attributes[['SAMPID', 'SUBJID', 'SMTSD']], on='SUBJID').drop(columns=['DTHHRDY'])
validsamples = gtexdf.columns
metadata = metadata[metadata['SAMPID'].isin(validsamples)]
metadata['SEX'] = metadata['SEX'] - 1

agemap = {'20-29': 0, '30-39': 1, '40-49': 2, '50-59': 3, '60-69': 4, '70-79': 5}
metadata['AGE'] = metadata['AGE'].map(agemap)

In [30]:
numtissues = len(set(metadata['SMTSD']))

In [31]:
numtissues

68

# FOR EACH TISSUE

In [32]:
tissue = "Brain - Cerebellum"

In [33]:
tissuesamples = metadata[metadata['SMTSD'] == tissue]
tissue_exp = gtexdf[list(tissuesamples['SAMPID'])]
tissue_exp_log = np.log2(tissue_exp + 1)

In [15]:
print(tissuesamples)

           SUBJID  SEX  AGE                    SAMPID               SMTSD  \
71     GTEX-111FC    0    4  GTEX-111FC-3326-SM-5GZYV  Brain - Cerebellum   
152    GTEX-1128S    1    4  GTEX-1128S-2826-SM-5N9DI  Brain - Cerebellum   
197    GTEX-117XS    0    4  GTEX-117XS-3126-SM-5GIDP  Brain - Cerebellum   
279    GTEX-1192X    0    3  GTEX-1192X-3226-SM-5987D  Brain - Cerebellum   
298    GTEX-11DXW    0    2  GTEX-11DXW-1026-SM-5H11K  Brain - Cerebellum   
...           ...  ...  ...                       ...                 ...   
19438   GTEX-ZVT3    1    4   GTEX-ZVT3-2926-SM-5GU6M  Brain - Cerebellum   
19512   GTEX-ZVZQ    1    4   GTEX-ZVZQ-2826-SM-HL9U2  Brain - Cerebellum   
19611   GTEX-ZYFD    0    3   GTEX-ZYFD-2926-SM-5GID9  Brain - Cerebellum   
19723   GTEX-ZYY3    1    4   GTEX-ZYY3-3026-SM-5GIEJ  Brain - Cerebellum   
19764   GTEX-ZZPT    0    3   GTEX-ZZPT-2926-SM-5EQ5S  Brain - Cerebellum   

       SMRIN  
71       7.1  
152      6.3  
197      7.2  
279      5.9  


In [16]:
highexp = ((tissue_exp_log > 0.5).sum(axis=1)) > 0.2 * tissue_exp.shape[1]
tissue_expdf = tissue_exp_log.loc[highexp]

In [17]:
tissue_exp_log.shape

(2000, 266)

In [18]:
tissue_exp_log

,GTEX-111FC-3326-SM-5GZYV,GTEX-1128S-2826-SM-5N9DI,GTEX-117XS-3126-SM-5GIDP,GTEX-1192X-3226-SM-5987D,GTEX-11DXW-1026-SM-5H11K,GTEX-11DXY-3126-SM-5N9BT,GTEX-11DYG-2926-SM-5H132,GTEX-11DZ1-2926-SM-5A5KI,GTEX-11EI6-2926-SM-5985U,GTEX-11EMC-3326-SM-5P9JH,...,GTEX-ZAJG-3026-SM-5HL92,GTEX-ZAK1-2926-SM-5HL9S,GTEX-ZDXO-2926-SM-4WKFM,GTEX-ZF28-2926-SM-4WKG1,GTEX-ZUA1-2926-SM-59HL3,GTEX-ZVT3-2926-SM-5GU6M,GTEX-ZVZQ-2826-SM-HL9U2,GTEX-ZYFD-2926-SM-5GID9,GTEX-ZYY3-3026-SM-5GIEJ,GTEX-ZZPT-2926-SM-5EQ5S
Name,,,,,,,,,,,,,,,,,,,,,
ENSG00000290825.2,0.040786,0.041669,0.000000,0.043759,0.044793,0.000000,0.000000,0.042359,0.026318,0.000000,...,0.000000,0.000000,0.042455,0.000000,0.000000,0.037745,0.000000,0.025200,0.167932,0.000000
ENSG00000223972.6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000310526.1,2.844231,2.493754,2.752630,2.895965,2.727923,2.721993,3.097899,2.362751,1.941213,2.722438,...,3.015664,2.406214,2.626580,2.692479,2.581239,2.318668,3.189719,2.353676,2.981486,2.033180
ENSG00000243485.6,0.000000,0.071484,0.000000,0.000000,0.000000,0.059849,0.068579,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.040968,0.138421,0.000000,0.137577,0.000000,0.060460,0.000000
ENSG00000237613.3,0.044082,0.000000,0.000000,0.000000,0.000000,0.000000,0.085121,0.045780,0.056363,0.000000,...,0.000000,0.044558,0.045883,0.000000,0.044645,0.040797,0.000000,0.000000,0.000000,0.045163
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000228929.1,0.873416,0.313747,1.009225,1.829058,1.121974,1.212063,1.189372,0.991356,0.937895,0.846755,...,1.032208,0.880587,1.162164,0.755964,2.205367,0.730534,0.780534,0.963106,1.282582,0.572664
ENSG00000206627.1,0.000000,0.610088,0.702205,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.604732,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000203995.11,0.482643,0.574941,0.312464,0.434458,0.401940,0.504083,0.610200,0.196242,0.275089,0.631798,...,0.359041,0.772994,0.652620,0.233960,0.616035,0.496196,0.435985,0.573658,0.498267,0.309790


# FOR EACH GENE

In [19]:
gene = "ENSG00000310526.1"

In [20]:
geneexp = tissue_expdf.loc[gene]

In [21]:
X = pd.DataFrame({'INTERCEPT': np.ones(len(tissuesamples)), 'AGE':tissuesamples['AGE'], 'SEX':tissuesamples['SEX'], 'SAMPID': tissuesamples['SAMPID'], 'SMRIN': tissuesamples['SMRIN']}).set_index('SAMPID')

In [22]:
model = sm.OLS(geneexp, X)
res = model.fit()

In [23]:
beta_age, t, p = res.params['AGE'], res.tvalues['AGE'], res.pvalues['AGE']

In [ ]:
beta_age

np.float64(-0.09561180974000315)

In [24]:
res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:      ENSG00000310526.1   R-squared:                       0.097
Model:                            OLS   Adj. R-squared:                  0.087
Method:                 Least Squares   F-statistic:                     9.415
Date:                Mon, 04 May 2026   Prob (F-statistic):           6.25e-06
Time:                        01:07:43   Log-Likelihood:                -194.73
No. Observations:                 266   AIC:                             397.5
Df Residuals:                     262   BIC:                             411.8
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
INTERCEPT      1.5833      0.363      4.361      0.000       0.868       2.298
AGE           -0.0956      0.030     -3.158      0.002      -0.155      -0.036
SEX            0.0317      0.070      0.453      0.651      -0.106       0.170
SMRIN          0.1923      0.050      3.863      0.000       0.094       0.290
==============================================================================
Omnibus:                        4.025   Durbin-Watson:                   1.589
Prob(Omnibus):                  0.134   Jarque-Bera (JB):                3.702
Skew:                           0.243   Prob(JB):                        0.157
Kurtosis:                       3.311   Cond. No.                         90.1
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [25]:
tissue_expdf.columns

Index(['GTEX-111FC-3326-SM-5GZYV', 'GTEX-1128S-2826-SM-5N9DI',
       'GTEX-117XS-3126-SM-5GIDP', 'GTEX-1192X-3226-SM-5987D',
       'GTEX-11DXW-1026-SM-5H11K', 'GTEX-11DXY-3126-SM-5N9BT',
       'GTEX-11DYG-2926-SM-5H132', 'GTEX-11DZ1-2926-SM-5A5KI',
       'GTEX-11EI6-2926-SM-5985U', 'GTEX-11EMC-3326-SM-5P9JH',
       ...
       'GTEX-ZAJG-3026-SM-5HL92', 'GTEX-ZAK1-2926-SM-5HL9S',
       'GTEX-ZDXO-2926-SM-4WKFM', 'GTEX-ZF28-2926-SM-4WKG1',
       'GTEX-ZUA1-2926-SM-59HL3', 'GTEX-ZVT3-2926-SM-5GU6M',
       'GTEX-ZVZQ-2826-SM-HL9U2', 'GTEX-ZYFD-2926-SM-5GID9',
       'GTEX-ZYY3-3026-SM-5GIEJ', 'GTEX-ZZPT-2926-SM-5EQ5S'],
      dtype='str', length=266)